In [ ]:
import snowflake
from snowflake.snowpark.context import get_active_session

session = get_active_session()

import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import pandas as pd


training_df = pd.read_csv('data/landsat/landsat_features_training_ross.csv')

submission_df = pd.read_csv('data/landsat/landsat_features_validation_ross.csv')

for col in training_df.columns:
    training_df = training_df.rename(columns={col: col.lower()})

training_df['sample date'] = pd.to_datetime(training_df['sample date'],format='%d-%m-%Y')

training_df.head(5)



for col in submission_df.columns:
    submission_df = submission_df.rename(columns={col: col.lower()})

submission_df['sample date'] = pd.to_datetime(submission_df['sample date'],format='%d-%m-%Y')


In [ ]:
training_df.columns

In [ ]:
B2	= 'blue'
B3	= 'green'
B4	= 'red'
B5	= 'nir08'
B6	= 'swir16'
B7	= 'swir22'


In [ ]:
original_set = [training_df, submission_df]

#original_set[i][B5]

for i in range(len(original_set)):

    B2	= original_set[i]['blue']
    B3	= original_set[i]['green']
    B4	= original_set[i]['red']
    B5	= original_set[i]['nir08']
    B6	= original_set[i]['swir16']
    B7	= original_set[i]['swir22']
    
    original_set[i]['evi'] = 2.5 * ((B5 - B4) / (B5 + 6*B4 - 7.5*B2 + 1))
    original_set[i]['osavi']	= 1.16 * (B5 - B4) / (B5 + B4 + 0.16)
    original_set[i]['gndvi']	= (B5 - B3) / (B5 + B3)
    original_set[i]['gcvi']	= (B5 / B3) - 1
    original_set[i]['msi']	= B6 / B5
    original_set[i]['bui']	= original_set[i]['ndbi'] - original_set[i]['nvdi']
    original_set[i]['tc brightness'] =	0.3029*B2 + 0.2786*B3 + 0.4733*B4 + 0.5599*B5 + 0.5080*B6 + 0.1872*B7
    original_set[i]['tc greenness'] =	-0.2941*B2 -0.2430*B3 -0.5424*B4 +0.7276*B5 +0.0713*B6 -0.1608*B7

    original_set[i]['nbr']=	(B5 - B7) / (B5 + B7)
    original_set[i]['ndsi']= 	(B3 - B6) / (B3 + B6)
    original_set[i]['green/red ratio'] =	B3 / B4
    original_set[i]['ndgi'] =	(B3 - B4) / (B3 + B4)
    original_set[i]['ui (urban index)'] =	((B6/B5) - (B4/B3))
    original_set[i]['nbr2'] =	(B6 - B7) / (B6 + B7)
    original_set[i]['red/nir ratio'] =	B4 / B5
    original_set[i]['green/nir ratio'] =	B3 / B5


In [ ]:
training_df.head(5)

In [ ]:
training_df.to_csv('landsat_features_training_mvdb.csv', index=False)
submission_df.to_csv('landsat_features_validation_mvdb.csv', index=False)

In [ ]:
session.sql("""
    PUT file://landsat_features_training_mvdb.csv
    'snow://workspace/USER$.PUBLIC."EY-AI-and-Data-Challenge-version2"/versions/live/data/landsat'
    AUTO_COMPRESS=FALSE
    OVERWRITE=TRUE
""").collect()

print("File saved! Refresh the browser to see the files in the sidebar")

In [ ]:
session.sql("""
    PUT file://landsat_features_validation_mvdb.csv
    'snow://workspace/USER$.PUBLIC."EY-AI-and-Data-Challenge-version2"/versions/live/data/landsat'
    AUTO_COMPRESS=FALSE
    OVERWRITE=TRUE
""").collect()

print("File saved! Refresh the browser to see the files in the sidebar")